In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os, cv2, numpy as np, pandas as pd
from tqdm import tqdm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121

2026-04-13 04:55:40.368231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776056140.560627      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776056140.616207      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776056141.035375      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776056141.035420      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776056141.035423      55 computation_placer.cc:177] computation placer alr

In [3]:
df = pd.read_csv("/kaggle/input/datasets/amimulahasanrofik/abu-csv/data_info.csv")
img_dir = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"

df['path'] = df['image_name'].apply(lambda x: os.path.join(img_dir, x))

In [4]:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1)

train_idx, test_idx = next(gss.split(df, groups=df['image_name']))

df_train = df.iloc[train_idx]
df_test = df.iloc[test_idx]

In [5]:
IMG_SIZE = 224

def load_images(df):
    imgs = []
    for path in tqdm(df['path']):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        imgs.append(img)
    return np.array(imgs)

X_img_train = load_images(df_train)
X_img_test = load_images(df_test)

100%|██████████| 1162/1162 [00:19<00:00, 60.30it/s]


In [6]:
tab_cols = ['age', 'gender', 'group']   # exclude thickness first

X_tab_train = df_train[tab_cols].values
X_tab_test = df_test[tab_cols].values

scaler = StandardScaler()
X_tab_train = scaler.fit_transform(X_tab_train)
X_tab_test = scaler.transform(X_tab_test)

In [7]:
le = LabelEncoder()
y_train = le.fit_transform(df_train['label'])
y_test = le.transform(df_test['label'])

num_classes = len(np.unique(y_train))

In [8]:
img_input = layers.Input(shape=(224,224,3))

base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_tensor=img_input
)

for layer in base_model.layers:
    layer.trainable = False

x1 = base_model.output
x1 = layers.GlobalAveragePooling2D()(x1)
x1 = layers.BatchNormalization()(x1)

I0000 00:00:1776056266.821156      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
tab_input = layers.Input(shape=(X_tab_train.shape[1],))

x2 = layers.Dense(32, activation='relu')(tab_input)
x2 = layers.BatchNormalization()(x2)
x2 = layers.Dropout(0.3)(x2)

In [10]:
combined = layers.Concatenate()([x1, x2])

In [11]:
x = layers.Dense(256, activation='relu')(combined)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.4)(x)

output = layers.Dense(num_classes, activation='softmax')(x)

In [12]:
model = models.Model(
    inputs=[img_input, tab_input],
    outputs=output
)

In [13]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=loss_fn,
    metrics=['accuracy']
)

TypeError: SparseCategoricalCrossentropy.__init__() got an unexpected keyword argument 'label_smoothing'

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3)
]

In [ ]:
history = model.fit(
    [X_img_train, X_tab_train],
    y_train,
    validation_data=([X_img_test, X_tab_test], y_test),
    epochs=30,
    batch_size=8,
    callbacks=callbacks
)

In [ ]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=loss_fn,
    metrics=['accuracy']
)

model.fit(
    [X_img_train, X_tab_train],
    y_train,
    validation_data=([X_img_test, X_tab_test], y_test),
    epochs=10,
    batch_size=8
)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict([X_img_test, X_tab_test])
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred))